In [ ]:
!pip -q install --upgrade openai huggingface_hub pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 146.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 8.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which

In [ ]:
 
# AIMES: MATCHED POSITIVE-NEGATIVE MFT GENERATION
# Version: v1
 

import os
import json
import time
import random
import pandas as pd
from datetime import datetime
from openai import OpenAI
from google.colab import userdata
from IPython.display import display

 
# 1. PATHS
 

AIMES_ROOT = "/content/drive/MyDrive/AIMES"

DATA_DIR = os.path.join(
    AIMES_ROOT,
    "data"
)

CONTRASTIVE_DIR = os.path.join(
    DATA_DIR,
    "contrastive"
)

os.makedirs(
    CONTRASTIVE_DIR,
    exist_ok=True
)

RAW_JSONL = os.path.join(
    CONTRASTIVE_DIR,
    "mft_positive_negative_raw_v1.jsonl"
)

FINAL_CSV = os.path.join(
    CONTRASTIVE_DIR,
    "mft_positive_negative_v1.csv"
)

FINAL_JSONL = os.path.join(
    CONTRASTIVE_DIR,
    "mft_positive_negative_v1.jsonl"
)

FINAL_JSON = os.path.join(
    CONTRASTIVE_DIR,
    "mft_positive_negative_v1.json"
)

CONFIG_JSON = os.path.join(
    CONTRASTIVE_DIR,
    "generation_config_v1.json"
)

SUMMARY_JSON = os.path.join(
    CONTRASTIVE_DIR,
    "dataset_summary_v1.json"
)

EXAMPLES_CSV = os.path.join(
    CONTRASTIVE_DIR,
    "example_positive_negative_pairs_v1.csv"
)

 
# 2. RESET V1
 

# True = delete and regenerate V1 from scratch
RESET_V1 = True

if RESET_V1:
    for path in [
        RAW_JSONL,
        FINAL_CSV,
        FINAL_JSONL,
        FINAL_JSON,
        CONFIG_JSON,
        SUMMARY_JSON,
        EXAMPLES_CSV
    ]:
        if os.path.exists(path):
            os.remove(path)
            print("Removed:", path)

 
# 3. OPENROUTER
 

OPENROUTER_API_KEY = userdata.get(
    "OPENROUTER_API_KEY"
)

assert OPENROUTER_API_KEY is not None, \
    "OPENROUTER_API_KEY not found in Colab Secrets."

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

GENERATOR_MODEL = "~openai/gpt-latest"

print("Generator:", GENERATOR_MODEL)

 
# 4. FOUNDATIONS
 

FOUNDATIONS = [
    "Care",
    "Fairness",
    "Loyalty",
    "Authority",
    "Sanctity"
]

FOUNDATION_POLES = {
    "Care": {
        "positive": "Care",
        "negative": "Harm"
    },
    "Fairness": {
        "positive": "Fairness",
        "negative": "Cheating"
    },
    "Loyalty": {
        "positive": "Loyalty",
        "negative": "Betrayal"
    },
    "Authority": {
        "positive": "Authority",
        "negative": "Subversion"
    },
    "Sanctity": {
        "positive": "Sanctity",
        "negative": "Degradation"
    }
}

 
# 5. DEFINITIONS
 

FOUNDATION_DEFINITIONS = {

    "Care": """
Care/Harm concerns compassion, protection from suffering,
emotional or physical well-being, cruelty, neglect, injury,
and responsiveness to another person's vulnerability.
The positive pole is Care and the negative pole is Harm.
""",

    "Fairness": """
Fairness/Cheating concerns justice, reciprocity, equal or
proportional treatment, honesty in exchanges, exploitation,
deception, cheating, and receiving what one deserves.
The positive pole is Fairness and the negative pole is Cheating.
""",

    "Loyalty": """
Loyalty/Betrayal concerns allegiance, solidarity, commitment
to one's group, family, team, organization, or community,
as well as desertion, betrayal, or placing personal interests
above important group commitments.
The positive pole is Loyalty and the negative pole is Betrayal.
""",

    "Authority": """
Authority/Subversion concerns legitimate hierarchy, leadership,
roles, duty, respect for legitimate authority, obedience,
defiance, insubordination, and undermining established authority.
The positive pole is Authority and the negative pole is Subversion.
""",

    "Sanctity": """
Sanctity/Degradation concerns purity, cleanliness, sacredness,
contamination, desecration, degradation, pollution, and disgust.
The positive pole is Sanctity and the negative pole is Degradation.
"""
}

 
# 6. DIVERSITY CONTEXTS
 

DIVERSITY_CONTEXTS = [
    "family and household",
    "friends and interpersonal relationships",
    "school and education",
    "workplace",
    "community and neighborhood",
    "groups, clubs, and organizations",
    "public and service interactions",
    "online and digital interactions",
    "sports and recreation",
    "health and caregiving",
    "travel and transportation",
    "social gatherings"
]

 
# 7. GENERATION SIZE
 

TARGET_PAIRS_PER_FOUNDATION = 200

# Generate a few matched pairs per API call
BATCH_SIZE = 5

TOTAL_EXPECTED_PAIRS = (
    len(FOUNDATIONS)
    * TARGET_PAIRS_PER_FOUNDATION
)

print(
    "Expected pairs:",
    TOTAL_EXPECTED_PAIRS
)

print(
    "Expected individual vignettes:",
    TOTAL_EXPECTED_PAIRS * 2
)

 
# 8. BALANCED CONTEXT SCHEDULE
 

def build_context_schedule(
    n_pairs,
    contexts,
    seed
):
    """
    Approximately equal allocation across all contexts.
    """
    rng = random.Random(seed)

    base = n_pairs // len(contexts)
    remainder = n_pairs % len(contexts)

    schedule = []

    for i, context in enumerate(contexts):

        n = base + (
            1 if i < remainder else 0
        )

        schedule.extend(
            [context] * n
        )

    rng.shuffle(schedule)

    assert len(schedule) == n_pairs

    return schedule


CONTEXT_SCHEDULES = {}

for i, foundation in enumerate(
    FOUNDATIONS
):
    CONTEXT_SCHEDULES[foundation] = (
        build_context_schedule(
            TARGET_PAIRS_PER_FOUNDATION,
            DIVERSITY_CONTEXTS,
            seed=1000 + i
        )
    )

 
# 9. SYSTEM PROMPT
 

SYSTEM_PROMPT = """
You are constructing a controlled research dataset for studying
Moral Foundations Theory representations in language models.

Generate MATCHED POSITIVE-NEGATIVE vignette pairs.

Each pair must describe essentially the same underlying social
situation in both versions.

The positive version must express the positive pole of the target
foundation. The negative version must express the negative pole.

MATCHING CONSTRAINTS:

1. Preserve the same actors across both versions.
2. Preserve the same social setting.
3. Preserve the same underlying event, opportunity, or decision.
4. Preserve approximately the same sentence structure.
5. Preserve approximately the same amount of detail.
6. Keep both versions similar in length.
7. Change only the behavior or decision needed to reverse the
   target moral polarity.
8. Do not add unrelated events to only one side.
9. Avoid making another Moral Foundations Theory dimension the
   primary difference between the two versions.
10. Avoid explicitly revealing the category with words such as
    "care", "harm", "fair", "unfair", "loyal", "disloyal",
    "authority", "subversion", "sanctity", "degradation",
    "moral", "immoral", "ethical", or "unethical" when those
    words merely reveal the intended label.
11. Do not explain why either behavior is morally good or bad.
12. Each vignette should normally be one concise sentence.
13. Each vignette should describe a plausible everyday event.
14. Avoid duplicate or trivial paraphrase pairs.
15. Return valid JSON only.

The goal is to vary moral polarity while keeping incidental context
as constant as possible.
"""

 
# 10. GENERATION PROMPT
 

def build_generation_prompt(
    foundation,
    context,
    n_pairs
):

    poles = FOUNDATION_POLES[
        foundation
    ]

    definition = FOUNDATION_DEFINITIONS[
        foundation
    ]

    return f"""
TARGET FOUNDATION:
{foundation}

POSITIVE POLE:
{poles["positive"]}

NEGATIVE POLE:
{poles["negative"]}

FOUNDATION DEFINITION:
{definition}

SOCIAL CONTEXT:
{context}

Generate exactly {n_pairs} matched positive-negative vignette pairs.

For each pair:

- keep the same actors,
- keep the same setting,
- keep the same underlying event or decision point,
- keep sentence structures closely matched,
- keep lengths approximately matched,
- make the primary difference the positive versus negative pole
  of {foundation},
- avoid explicitly naming the foundation or pole,
- avoid introducing another moral foundation as the primary contrast,
- use natural and concrete everyday language,
- preferably use roughly 10-30 words per vignette.

Return exactly:

{{
  "foundation": "{foundation}",
  "context": "{context}",
  "pairs": [
    {{
      "positive_text": "...",
      "negative_text": "..."
    }}
  ]
}}

The pairs array must contain exactly {n_pairs} items.
"""

 
# 11. GENERATOR CALL
 

def call_generator(
    foundation,
    context,
    n_pairs,
    max_retries=5
):

    prompt = build_generation_prompt(
        foundation,
        context,
        n_pairs
    )

    for attempt in range(
        max_retries
    ):

        try:

            response = (
                client.chat.completions.create(
                    model=GENERATOR_MODEL,

                    messages=[
                        {
                            "role": "system",
                            "content": SYSTEM_PROMPT
                        },
                        {
                            "role": "user",
                            "content": prompt
                        }
                    ],

                    temperature=0.7,

                    response_format={
                        "type": "json_object"
                    }
                )
            )

            content = (
                response
                .choices[0]
                .message
                .content
            )

            return json.loads(
                content
            )

        except Exception as e:

            print(
                f"Attempt "
                f"{attempt + 1}/"
                f"{max_retries} failed:"
            )

            print(e)

            if attempt < (
                max_retries - 1
            ):
                time.sleep(
                    5 * (
                        attempt + 1
                    )
                )

    raise RuntimeError(
        f"Failed generation: "
        f"{foundation} / {context}"
    )

 
# 12. STRUCTURAL CHECK
 

def validate_generated_batch(
    data,
    foundation,
    context,
    expected_n
):

    if not isinstance(
        data,
        dict
    ):
        return False

    if data.get(
        "foundation"
    ) != foundation:
        return False

    if data.get(
        "context"
    ) != context:
        return False

    pairs = data.get(
        "pairs"
    )

    if not isinstance(
        pairs,
        list
    ):
        return False

    if len(
        pairs
    ) != expected_n:
        return False

    for pair in pairs:

        if not isinstance(
            pair,
            dict
        ):
            return False

        pos = pair.get(
            "positive_text"
        )

        neg = pair.get(
            "negative_text"
        )

        if not isinstance(
            pos,
            str
        ):
            return False

        if not isinstance(
            neg,
            str
        ):
            return False

        pos = pos.strip()
        neg = neg.strip()

        if len(pos) < 10:
            return False

        if len(neg) < 10:
            return False

        if pos.lower() == neg.lower():
            return False

    return True

 
# 13. JSONL HELPERS
 

def append_jsonl(
    records,
    filename
):

    with open(
        filename,
        "a",
        encoding="utf-8"
    ) as f:

        for record in records:

            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                )
                + "\n"
            )


def load_jsonl(
    filename
):

    records = []

    if not os.path.exists(
        filename
    ):
        return records

    with open(
        filename,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if line:
                records.append(
                    json.loads(
                        line
                    )
                )

    return records

 
# 14. GENERATE
 

existing_records = load_jsonl(
    RAW_JSONL
)

counts = {
    foundation: 0
    for foundation in FOUNDATIONS
}

for record in existing_records:

    foundation = record.get(
        "foundation"
    )

    if foundation in counts:
        counts[foundation] += 1

print(
    "\nStarting counts:"
)

print(
    counts
)

for foundation in FOUNDATIONS:

    print(
        "\n"
        + "=" * 90
    )

    print(
        "Generating:",
        foundation
    )

    print(
        "=" * 90
    )

    current_count = counts[
        foundation
    ]

    schedule = CONTEXT_SCHEDULES[
        foundation
    ]

    while (
        current_count
        <
        TARGET_PAIRS_PER_FOUNDATION
    ):

        context = schedule[
            current_count
        ]

        remaining = (
            TARGET_PAIRS_PER_FOUNDATION
            -
            current_count
        )

        max_batch = min(
            BATCH_SIZE,
            remaining
        )

        # Only batch consecutive items sharing the same context
        n_this_batch = 1

        while (
            n_this_batch < max_batch
            and
            schedule[
                current_count
                +
                n_this_batch
            ]
            ==
            context
        ):
            n_this_batch += 1

        print(
            f"{foundation}: "
            f"{current_count}/"
            f"{TARGET_PAIRS_PER_FOUNDATION}"
            f" | {context}"
            f" | batch={n_this_batch}"
        )

        batch = call_generator(
            foundation,
            context,
            n_this_batch
        )

        if not validate_generated_batch(
            batch,
            foundation,
            context,
            n_this_batch
        ):

            print(
                "Invalid batch. Regenerating."
            )

            continue

        new_records = []

        for pair in batch[
            "pairs"
        ]:

            new_records.append(
                {
                    "foundation":
                        foundation,

                    "positive_pole":
                        FOUNDATION_POLES[
                            foundation
                        ]["positive"],

                    "negative_pole":
                        FOUNDATION_POLES[
                            foundation
                        ]["negative"],

                    "context":
                        context,

                    "positive_text":
                        pair[
                            "positive_text"
                        ].strip(),

                    "negative_text":
                        pair[
                            "negative_text"
                        ].strip(),

                    "generator":
                        GENERATOR_MODEL,

                    "generation_time":
                        datetime.now().isoformat(),

                    "generation_version":
                        "v1"
                }
            )

        append_jsonl(
            new_records,
            RAW_JSONL
        )

        current_count += len(
            new_records
        )

        counts[
            foundation
        ] = current_count

        time.sleep(1)

print(
    "\nGeneration complete."
)

print(
    counts
)

 
# 15. LOAD GENERATED DATA
 

records = load_jsonl(
    RAW_JSONL
)

df = pd.DataFrame(
    records
)

print(
    "\nRaw shape:",
    df.shape
)

print(
    df[
        "foundation"
    ].value_counts()
)

 
# 16. DUPLICATE CHECK
 

df["_pos_norm"] = (
    df[
        "positive_text"
    ]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)

df["_neg_norm"] = (
    df[
        "negative_text"
    ]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)

duplicate_pairs = df.duplicated(
    subset=[
        "foundation",
        "_pos_norm",
        "_neg_norm"
    ],
    keep="first"
)

print(
    "\nExact duplicate pairs:",
    duplicate_pairs.sum()
)

df = df.loc[
    ~duplicate_pairs
].copy()

df.drop(
    columns=[
        "_pos_norm",
        "_neg_norm"
    ],
    inplace=True
)

df.reset_index(
    drop=True,
    inplace=True
)

 
# 17. ASSIGN PAIR IDS
 

FOUNDATION_CODES = {
    "Care": "care",
    "Fairness": "fair",
    "Loyalty": "loyal",
    "Authority": "auth",
    "Sanctity": "sanct"
}

df[
    "pair_id"
] = ""

for foundation in FOUNDATIONS:

    mask = (
        df[
            "foundation"
        ]
        ==
        foundation
    )

    indices = df.index[
        mask
    ]

    prefix = FOUNDATION_CODES[
        foundation
    ]

    ids = [
        f"{prefix}_{i:04d}"
        for i in range(
            1,
            len(indices) + 1
        )
    ]

    df.loc[
        indices,
        "pair_id"
    ] = ids

 
# 18. ORDER COLUMNS
 

df = df[
    [
        "pair_id",
        "foundation",
        "positive_pole",
        "negative_pole",
        "context",
        "positive_text",
        "negative_text",
        "generator",
        "generation_time",
        "generation_version"
    ]
]

 
# 19. SAVE FINAL DATA
 

df.to_csv(
    FINAL_CSV,
    index=False
)

df.to_json(
    FINAL_JSONL,
    orient="records",
    lines=True,
    force_ascii=False
)

df.to_json(
    FINAL_JSON,
    orient="records",
    indent=2,
    force_ascii=False
)

 
# 20. SAVE CONFIG
 

generation_config = {
    "project":
        "AIMES",

    "version":
        "v1",

    "dataset":
        "matched positive-negative MFT corpus",

    "generator":
        GENERATOR_MODEL,

    "temperature":
        0.7,

    "foundations":
        FOUNDATIONS,

    "foundation_poles":
        FOUNDATION_POLES,

    "foundation_definitions":
        FOUNDATION_DEFINITIONS,

    "diversity_contexts":
        DIVERSITY_CONTEXTS,

    "pairs_per_foundation":
        TARGET_PAIRS_PER_FOUNDATION,

    "total_expected_pairs":
        TOTAL_EXPECTED_PAIRS,

    "total_expected_vignettes":
        TOTAL_EXPECTED_PAIRS * 2,

    "batch_size":
        BATCH_SIZE,

    "generation_design":
        "positive and negative members generated jointly",

    "context_design":
        "approximately balanced across 12 diversity contexts",

    "system_prompt":
        SYSTEM_PROMPT,

    "dataset_role":
        "primary AIMES activation-direction construction"
}

with open(
    CONFIG_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        generation_config,
        f,
        indent=2,
        ensure_ascii=False
    )

 
# 21. SAVE SUMMARY
 

summary = {
    "total_pairs":
        int(len(df)),

    "total_vignettes":
        int(len(df) * 2),

    "pairs_by_foundation": {
        foundation:
            int(
                (
                    df[
                        "foundation"
                    ]
                    ==
                    foundation
                ).sum()
            )
        for foundation in FOUNDATIONS
    },

    "context_counts": {
        foundation:
            df[
                df[
                    "foundation"
                ]
                ==
                foundation
            ][
                "context"
            ]
            .value_counts()
            .to_dict()
        for foundation in FOUNDATIONS
    }
}

with open(
    SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )

 
# 22. PRINT ONE PAIR PER FOUNDATION
 

example_rows = []

for i, foundation in enumerate(
    FOUNDATIONS
):

    subset = df[
        df[
            "foundation"
        ]
        ==
        foundation
    ]

    example = (
        subset
        .sample(
            1,
            random_state=100 + i
        )
        .iloc[0]
    )

    example_rows.append(
        {
            "Foundation":
                foundation,

            "Context":
                example[
                    "context"
                ],

            "Positive Pole":
                example[
                    "positive_pole"
                ],

            "Positive Example":
                example[
                    "positive_text"
                ],

            "Negative Pole":
                example[
                    "negative_pole"
                ],

            "Negative Example":
                example[
                    "negative_text"
                ]
        }
    )

example_df = pd.DataFrame(
    example_rows
)

print(
    "\n"
    + "=" * 100
)

print(
    "ONE MATCHED PAIR PER FOUNDATION"
)

print(
    "=" * 100
)

display(
    example_df.style
    .set_properties(
        **{
            "text-align":
                "left",

            "white-space":
                "normal",

            "vertical-align":
                "top"
        }
    )
)

example_df.to_csv(
    EXAMPLES_CSV,
    index=False
)

 
# 23. PRINT EACH PAIR SIDE-BY-SIDE
 

for _, row in example_df.iterrows():

    print(
        "\n"
        + "-" * 100
    )

    print(
        row["Foundation"],
        ":",
        row["Positive Pole"],
        "vs.",
        row["Negative Pole"]
    )

    print(
        "Context:",
        row["Context"]
    )

    print(
        "-" * 100
    )

    pair_df = pd.DataFrame(
        {
            row[
                "Positive Pole"
            ]: [
                row[
                    "Positive Example"
                ]
            ],

            row[
                "Negative Pole"
            ]: [
                row[
                    "Negative Example"
                ]
            ]
        }
    )

    display(
        pair_df.style
        .set_properties(
            **{
                "text-align":
                    "left",

                "white-space":
                    "normal",

                "vertical-align":
                    "top"
            }
        )
    )

 
# 24. CONTEXT BALANCE
 

print(
    "\nContext distribution:"
)

context_table = pd.crosstab(
    df["context"],
    df["foundation"]
)

display(
    context_table
)

 
# 25. FINAL FILES
 

print(
    "\nFiles saved:"
)

for filename in sorted(
    os.listdir(
        CONTRASTIVE_DIR
    )
):
    print(
        filename
    )

print(
    "\nContrastive V1 generation complete."
)